Experiment with deep NNs architecture for text classification task.



In [ ]:
#I used 'hate' dataset from Hugging Face.
#so for this task I also load the same dataset, but to see how deep NNs will improve classification

import pandas as pd
import re
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

dataset = load_dataset('cardiffnlp/tweet_eval','hate')
print(dataset)

# Goal: to train model to find hater's tweets. For example, this is useful for users, they can block haters, filter content and etc
# Task described: to teach model to split tweets on hate or non-hate ones

train_df = pd.DataFrame(dataset['train'])
val_df = pd.DataFrame(dataset['validation'])
test_df = pd.DataFrame(dataset['test'])

texts_train = train_df['text'].tolist()

#basic preprocessing: cleaning (remove links, hashtags, mentions)
def re_clean(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'[^A-Za-z\s]', '', text)
    return text.lower().strip()

# 2) Convert texts to bow representation
from sklearn.feature_extraction.text import CountVectorizer

# applied cleaning to each split:
train_clean = [re_clean(t) for t in train_df['text']]
val_clean   = [re_clean(t) for t in val_df['text']]
test_clean  = [re_clean(t) for t in test_df['text']]

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

vectorizer = CountVectorizer(
    max_features=10000,   #limited vocab size to avoid huge matrices
    stop_words='english',
)
vectorizer.fit(train_clean)

X_train_bow = vectorizer.transform(train_clean)
X_val_bow   = vectorizer.transform(val_clean)
X_test_bow  = vectorizer.transform(test_clean)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

hate/train-00000-of-00001.parquet:   0%|          | 0.00/816k [00:00<?, ?B/s]

hate/test-00000-of-00001.parquet:   0%|          | 0.00/278k [00:00<?, ?B/s]

hate/validation-00000-of-00001.parquet:   0%|          | 0.00/103k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2970 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2970
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1000
    })
})


In [ ]:
# 3) Train neural network

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd


# dataset
class SparseTFIDFDataset(Dataset):
    def __init__(self, X_csr, y):
        self.X = X_csr.tocsr()
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx].toarray()[0], dtype=torch.float32)
        return x, self.y[idx]

# MLP model
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes,
                 activation="relu", dropout=0.6):
        super().__init__()

        activations = {
            "relu": nn.ReLU(),
            "gelu": nn.GELU()
        }
        if activation not in activations:
            raise ValueError("Activation must be: relu, gelu")

        self.activation = activations[activation]

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.dropout(x)
        x = self.activation(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)

# train
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    n, correct, running_loss = 0, 0, 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        n += yb.size(0)
        running_loss += loss.item() * yb.size(0)

    return correct / n, running_loss / n


# evaluation
@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    model.eval()
    all_true, all_pred = [], []
    total_loss, n = 0.0, 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)

        if criterion:
            total_loss += criterion(logits, yb).item() * yb.size(0)
            n += yb.size(0)

        pred = logits.argmax(dim=1)
        all_true.append(yb.cpu())
        all_pred.append(pred.cpu())

    y_true = torch.cat(all_true).numpy()
    y_pred = torch.cat(all_pred).numpy()
    acc = accuracy_score(y_true, y_pred)
    loss = total_loss / n if criterion else None
    return acc, y_true, y_pred, loss


# general experiment engine
def run_experiment(
        param_list,
        param_name,
        model_args_base,
        optimizer_fn,
        train_loader, val_loader, test_loader,
        num_epochs=20, patience=5):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    criterion = nn.CrossEntropyLoss()
    results = []

    MODEL_PARAMS = {"input_size", "hidden_size", "num_classes", "activation", "dropout"}

    for param in param_list:
        print(f"\n------ Running: {param_name} = {param} ------")

        # update varying model parameters
        model_args = model_args_base.copy()

        if param_name in MODEL_PARAMS:
           model_args[param_name] = param


        model = MLP(**model_args).to(device)
        optimizer = optimizer_fn(model.parameters(), param)

        best_val_acc = -1
        patience_counter = 0

        for epoch in range(1, num_epochs + 1):
            train_acc, _ = train_one_epoch(model, train_loader, criterion, optimizer, device)
            val_acc, _, _, _ = evaluate(model, val_loader, device, criterion)

            print(f"Epoch {epoch:02d} | Train {train_acc:.4f} | Val {val_acc:.4f}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break

        test_acc, _, _, _ = evaluate(model, test_loader, device)

        results.append({
            param_name: param,
            "Best Val Acc": best_val_acc,
            "Test Acc": test_acc
        })

        print(f"{param_name}={param} | Best Val={best_val_acc:.4f} | Test={test_acc:.4f}")

    return pd.DataFrame(results)


In [ ]:
train_dataset = SparseTFIDFDataset(X_train_bow, y_train)
val_dataset   = SparseTFIDFDataset(X_val_bow, y_val)
test_dataset  = SparseTFIDFDataset(X_test_bow, y_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=128, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False)

input_size = X_train_bow.shape[1]
num_classes = len(np.unique(y_train))


# I need to make experiments in order to see how they influence quality

In this part I will experiment with:

the number of hidden layers (32, 64, 128, 256, 500)
activation functions(ReLU, GELU)
optimizers (AdamW, Adam, SGD + momentum)
learning rate adjustments (1e-3, 5e-4, 1e-4, 5e-5)
dropout (0.5, 0.7, 0.9)



**1. Experiment with the number of hidden layers (32, 64, 128, 256, 500)**

In [ ]:
# hidden size experiments

hidden_sizes = [32, 64, 128, 256, 500]

results_hidden = run_experiment(
    param_list=hidden_sizes,
    param_name="hidden_size",
    model_args_base=dict(input_size=input_size, hidden_size=128,
                         num_classes=num_classes, activation="relu", dropout=0.6),
    optimizer_fn=lambda params, _: torch.optim.AdamW(params, lr=1e-4, weight_decay=5e-3),
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    num_epochs=20,
    patience=5
)

results_hidd_size_df = pd.DataFrame(results_hidden)
print("\n------ Comparative Table ------")
print(results_hidd_size_df)



------ Running: hidden_size = 32 ------
Epoch 01 | Train 0.4212 | Val 0.4270
Epoch 02 | Train 0.4251 | Val 0.4270
Epoch 03 | Train 0.4436 | Val 0.4270
Epoch 04 | Train 0.4866 | Val 0.4670
Epoch 05 | Train 0.5527 | Val 0.6710
Epoch 06 | Train 0.6020 | Val 0.6800
Epoch 07 | Train 0.6588 | Val 0.6840
Epoch 08 | Train 0.7117 | Val 0.6860
Epoch 09 | Train 0.7396 | Val 0.6830
Epoch 10 | Train 0.7607 | Val 0.6800
Epoch 11 | Train 0.7668 | Val 0.6810
Epoch 12 | Train 0.7891 | Val 0.6820
Epoch 13 | Train 0.7951 | Val 0.6820
hidden_size=32 | Best Val=0.6860 | Test=0.5650

------ Running: hidden_size = 64 ------
Epoch 01 | Train 0.5797 | Val 0.5730
Epoch 02 | Train 0.5797 | Val 0.5730
Epoch 03 | Train 0.5797 | Val 0.5730
Epoch 04 | Train 0.5821 | Val 0.5800
Epoch 05 | Train 0.6122 | Val 0.6360
Epoch 06 | Train 0.6884 | Val 0.6710
Epoch 07 | Train 0.7474 | Val 0.6870
Epoch 08 | Train 0.7807 | Val 0.6850
Epoch 09 | Train 0.7987 | Val 0.6840
Epoch 10 | Train 0.8166 | Val 0.6850
Epoch 11 | Train 0.8

**2. Activation functions(ReLU, GELU)**

In [ ]:
# activation function experiments

activations = ["relu", "gelu"]

results_activation = run_experiment(
    param_list=activations,
    param_name="activation",
    model_args_base=dict(input_size=input_size, hidden_size=128,
                         num_classes=num_classes, activation="relu", dropout=0.6),
    optimizer_fn=lambda params, _: torch.optim.AdamW(params, lr=1e-4, weight_decay=5e-3),
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    num_epochs=30,
    patience=5
)

# comparative table
results_activation_df = pd.DataFrame(results_activation)
print("\n------ Comparative Table ------")
print(results_activation_df)



------ Running: activation = relu ------
Epoch 01 | Train 0.5579 | Val 0.5730
Epoch 02 | Train 0.5783 | Val 0.5730
Epoch 03 | Train 0.5850 | Val 0.5790
Epoch 04 | Train 0.6269 | Val 0.6620
Epoch 05 | Train 0.7279 | Val 0.6850
Epoch 06 | Train 0.7861 | Val 0.6760
Epoch 07 | Train 0.8147 | Val 0.6760
Epoch 08 | Train 0.8336 | Val 0.6760
Epoch 09 | Train 0.8521 | Val 0.6830
Epoch 10 | Train 0.8642 | Val 0.6850
activation=relu | Best Val=0.6850 | Test=0.5747

------ Running: activation = gelu ------
Epoch 01 | Train 0.4824 | Val 0.5860
Epoch 02 | Train 0.5988 | Val 0.5860
Epoch 03 | Train 0.6057 | Val 0.6210
Epoch 04 | Train 0.6747 | Val 0.6660
Epoch 05 | Train 0.7389 | Val 0.6830
Epoch 06 | Train 0.7792 | Val 0.6850
Epoch 07 | Train 0.8034 | Val 0.6830
Epoch 08 | Train 0.8186 | Val 0.6890
Epoch 09 | Train 0.8317 | Val 0.6810
Epoch 10 | Train 0.8439 | Val 0.6780
Epoch 11 | Train 0.8577 | Val 0.6730
Epoch 12 | Train 0.8670 | Val 0.6810
Epoch 13 | Train 0.8776 | Val 0.6780
activation=gelu |

**3. Optimizers (AdamW, Adam, SGD + momentum)**

In [ ]:
optimizers = {
    "AdamW": lambda params: torch.optim.AdamW(params, lr=1e-4, weight_decay=5e-3),
    "Adam": lambda params: torch.optim.Adam(params, lr=1e-4, weight_decay=5e-3),
    "SGD_momentum": lambda params: torch.optim.SGD(params, lr=1e-2, momentum=0.9, weight_decay=5e-3)
}

results_optim = []

for opt_name in optimizers.keys():

    df = run_experiment(
        param_list=[opt_name],
        param_name="optimizer",
        model_args_base=dict(
            input_size=input_size,
            hidden_size=128,
            num_classes=num_classes,
            activation="relu",
            dropout=0.6
        ),
        optimizer_fn=lambda params, name=opt_name: optimizers[name](params),
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        num_epochs=30,
        patience=5
    )

    results_optim.append(df)

results_optimizer_df = pd.concat(results_optim, ignore_index=True)
print("\n------ Comparative Table ------")
print(results_optimizer_df)



------ Running: optimizer = AdamW ------
Epoch 01 | Train 0.5739 | Val 0.5730
Epoch 02 | Train 0.5804 | Val 0.5730
Epoch 03 | Train 0.5809 | Val 0.5760
Epoch 04 | Train 0.6193 | Val 0.6610
Epoch 05 | Train 0.7227 | Val 0.6830
Epoch 06 | Train 0.7850 | Val 0.6870
Epoch 07 | Train 0.8167 | Val 0.6790
Epoch 08 | Train 0.8332 | Val 0.6810
Epoch 09 | Train 0.8489 | Val 0.6740
Epoch 10 | Train 0.8681 | Val 0.6830
Epoch 11 | Train 0.8789 | Val 0.6780
optimizer=AdamW | Best Val=0.6870 | Test=0.5643

------ Running: optimizer = Adam ------
Epoch 01 | Train 0.5634 | Val 0.5730
Epoch 02 | Train 0.5816 | Val 0.5730
Epoch 03 | Train 0.5806 | Val 0.5750
Epoch 04 | Train 0.5936 | Val 0.6140
Epoch 05 | Train 0.6523 | Val 0.6620
Epoch 06 | Train 0.7213 | Val 0.6760
Epoch 07 | Train 0.7631 | Val 0.6840
Epoch 08 | Train 0.7892 | Val 0.6820
Epoch 09 | Train 0.8069 | Val 0.6760
Epoch 10 | Train 0.8173 | Val 0.6780
Epoch 11 | Train 0.8273 | Val 0.6740
Epoch 12 | Train 0.8389 | Val 0.6730
optimizer=Adam | B

**4. Learning rate adjustments (1e-3, 5e-4, 1e-4, 5e-5)**

In [ ]:
learning_rates = [1e-3, 5e-4, 1e-4, 5e-5]

results_lr = run_experiment(
    param_list=learning_rates,
    param_name="lr",
    model_args_base=dict(input_size=input_size, hidden_size=128,
                         num_classes=num_classes, activation="relu", dropout=0.6),
    optimizer_fn=lambda params, lr: torch.optim.AdamW(params, lr=lr, weight_decay=5e-3),
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    num_epochs=30,
    patience=5
)

#comparative table
results_learn_rate_df = pd.DataFrame(results_lr)
print("\n------ Comparative Table ------")
print(results_learn_rate_df)




------ Running: lr = 0.001 ------
Epoch 01 | Train 0.6262 | Val 0.6760
Epoch 02 | Train 0.7888 | Val 0.6820
Epoch 03 | Train 0.8496 | Val 0.6610
Epoch 04 | Train 0.8879 | Val 0.6800
Epoch 05 | Train 0.9180 | Val 0.6640
Epoch 06 | Train 0.9376 | Val 0.6820
Epoch 07 | Train 0.9557 | Val 0.6780
lr=0.001 | Best Val=0.6820 | Test=0.5727

------ Running: lr = 0.0005 ------
Epoch 01 | Train 0.5692 | Val 0.6020
Epoch 02 | Train 0.7293 | Val 0.6840
Epoch 03 | Train 0.8168 | Val 0.6750
Epoch 04 | Train 0.8587 | Val 0.6760
Epoch 05 | Train 0.8929 | Val 0.6770
Epoch 06 | Train 0.9132 | Val 0.6610
Epoch 07 | Train 0.9320 | Val 0.6720
lr=0.0005 | Best Val=0.6840 | Test=0.5801

------ Running: lr = 0.0001 ------
Epoch 01 | Train 0.4449 | Val 0.4670
Epoch 02 | Train 0.5879 | Val 0.5740
Epoch 03 | Train 0.5986 | Val 0.6030
Epoch 04 | Train 0.6403 | Val 0.6600
Epoch 05 | Train 0.7288 | Val 0.6820
Epoch 06 | Train 0.7803 | Val 0.6840
Epoch 07 | Train 0.8158 | Val 0.6800
Epoch 08 | Train 0.8324 | Val 0.6

**5. Dropout (0.5, 0.7, 0.9)**

In [ ]:
dropouts = [0.5, 0.7, 0.9]

results_dropout = run_experiment(
    param_list=dropouts,
    param_name="dropout",
    model_args_base=dict(input_size=input_size, hidden_size=128,
                         num_classes=num_classes, activation="relu", dropout=0.6),
    optimizer_fn=lambda params, _: torch.optim.AdamW(params, lr=1e-4, weight_decay=5e-3),
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    num_epochs=30,
    patience=5
)

# comparative table
results_dropout_df = pd.DataFrame(results_dropout)
print("\n------ Comparative Table ------")
print(results_dropout_df)



------ Running: dropout = 0.5 ------
Epoch 01 | Train 0.5120 | Val 0.5730
Epoch 02 | Train 0.5899 | Val 0.5730
Epoch 03 | Train 0.5967 | Val 0.6180
Epoch 04 | Train 0.6876 | Val 0.6700
Epoch 05 | Train 0.7753 | Val 0.6850
Epoch 06 | Train 0.8119 | Val 0.6800
Epoch 07 | Train 0.8343 | Val 0.6820
Epoch 08 | Train 0.8550 | Val 0.6720
Epoch 09 | Train 0.8722 | Val 0.6730
Epoch 10 | Train 0.8848 | Val 0.6750
dropout=0.5 | Best Val=0.6850 | Test=0.5650

------ Running: dropout = 0.7 ------
Epoch 01 | Train 0.5761 | Val 0.5730
Epoch 02 | Train 0.5809 | Val 0.5730
Epoch 03 | Train 0.5803 | Val 0.5720
Epoch 04 | Train 0.5892 | Val 0.6040
Epoch 05 | Train 0.6503 | Val 0.6710
Epoch 06 | Train 0.7400 | Val 0.6840
Epoch 07 | Train 0.7857 | Val 0.6810
Epoch 08 | Train 0.8048 | Val 0.6760
Epoch 09 | Train 0.8207 | Val 0.6790
Epoch 10 | Train 0.8359 | Val 0.6800
Epoch 11 | Train 0.8493 | Val 0.6840
dropout=0.7 | Best Val=0.6840 | Test=0.5599

------ Running: dropout = 0.9 ------
Epoch 01 | Train 0.56

**Table with final results**

In [ ]:
# 5)final results: table with experiments (hyperparameter settings which were changed and quality) + conclusion (how differ from classical ML).

import pandas as pd

hidden_size_results_df = unify_results(hidden_size_results_df, 'hidden_size', 'Hidden Size')
activation_results_df = unify_results(activation_results_df, 'activation', 'Activation')
optimizer_results_df = unify_results(optimizer_results_df, 'optimizer', 'Optimizer')
lr_results_df = unify_results(lr_results_df, 'lr', 'Learning Rate')
dropout_results_df = unify_results(dropout_results_df, 'dropout', 'Dropout')


def unify_results(df, col_name, hyper_name):
    df = df.copy()
    df['Hyperparameter'] = hyper_name
    df = df.rename(columns={col_name: 'Value'})
    return df[['Hyperparameter', 'Value', 'Best Val Acc', 'Test Acc']]

hidden_size_results_df = unify_results(hidden_size_results_df, 'Hidden size', 'Hidden Size')
activation_results_df = unify_results(activation_results_df, 'Activation', 'Activation')
optimizer_results_df = unify_results(optimizer_results_df, 'Optimizer', 'Optimizer')
lr_results_df = unify_results(lr_results_df, 'Learning Rate', 'Learning Rate')
dropout_results_df = unify_results(dropout_results_df, 'Dropout', 'Dropout')


# table 1 -> grouped by hyperparameter type

grouped_table = pd.concat([
    hidden_size_results_df,
    activation_results_df,
    optimizer_results_df,
    lr_results_df,
    dropout_results_df
], ignore_index=True)

grouped_table.reset_index(drop=True, inplace=True)
print("------ Table 1: Grouped by hyperparameter ------\n")
display(grouped_table)

# Table 2 -> sorted by test accuracy

sorted_table = grouped_table.sort_values(by='Test Acc', ascending=False).reset_index(drop=True)
print("\n\n------ Table 2: Sorted by test accuracy ------\n")
display(sorted_table)


------ Table 1: Grouped by hyperparameter ------



,Hyperparameter,Value,Best Val Acc,Test Acc
0,Hidden Size,32,0.686,0.564983
1,Hidden Size,64,0.687,0.565657
2,Hidden Size,128,0.686,0.565993
3,Hidden Size,256,0.687,0.564983
4,Hidden Size,500,0.687,0.587879
5,Activation,relu,0.683,0.566667
6,Activation,gelu,0.688,0.585859
7,Optimizer,AdamW,0.687,0.564310
8,Optimizer,Adam,0.684,0.570034
9,Optimizer,SGD_momentum,0.686,0.545455




------ Table 2: Sorted by test accuracy ------



,Hyperparameter,Value,Best Val Acc,Test Acc
0,Hidden Size,500,0.687,0.587879
1,Activation,gelu,0.688,0.585859
2,Learning Rate,0.0005,0.684,0.580135
3,Dropout,0.9,0.573,0.578451
4,Learning Rate,0.001,0.682,0.572727
5,Optimizer,Adam,0.684,0.570034
6,Activation,relu,0.683,0.566667
7,Hidden Size,128,0.686,0.565993
8,Hidden Size,64,0.687,0.565657
9,Hidden Size,32,0.686,0.564983


# Conclusion:
According to the experiments, unlike classical ML, where models (linear regression,decision trees,SVMs) often have few hyperparameters and predictable effects, tuning a neural network involves a larger and more complex hyperparameters, including hidden layer size, activation functions, optimizers, learning rates and dropout.

With increase of hidden size, is does not always lead to better test accuracy. The best performance occurs at 500 hidd size (test accuracy = 0.580), but smaller hidd sizes like 64 also perform pretty well, so these highlights non linear interactions between architecture and training.

In activation function GELU showed better result than ReLU, which highlights that activation selection can have a strong impact on final performance of model

The best optimizer is Adam, but not too better than AdamW

In learning rate 0.001 is better then others, so bigger learnung rate provide ctronger results in this model.

As for dropout, regularization should not be too strong or too weak, the middle number 0.7 shows the best result. We need to balance model capacity and regularization. This in classical ML rarely requires.

The highest test accuracy was reached by GELU activation function (0.589), by learning rahe 0.001 (0.582), and by hidden size 500(0.580). I suppose that combination of these parameters could lead to even better test accuracy.
Classical ML models typically follow more predictable patterns ->  better feature scaling or regularization usually lead to consistent performance improvements.

All in all, this process demonstrates that deep learning hyperparameters changing is more experimental and less deterministic than in classical ML. While classical ML can often rely on domain knowledge and analytical reasoning for hyperparameter choices, neural networks require systematic exploration and careful evaluation across multiple axes, emphasizing the importance of structured experiments and performance driven selection.




I will need to fine tune existing models from hugging face for very same task. Tried several models of differnet size, older and newer ones.


In [ ]:
# Fine tuning existing models for the same task of classification

# Models chosen for this task:
#     1) distilbert-base-uncased (very small)
#     2) bert-base-uncased (baseline) google-bert/bert-base-uncased
#     3) roberta-base (stronger baseline)
#     4) microsoft/deberta-v3-base (new strong model)

# for this task changed from CPU to GPU T4, otherwise these models did not work

# Initialization cell (run once)

#!pip install -q transformers datasets evaluate

# I varied not too much of params and in different cells, othervise it takes it takes ages to process
# Moreover, I run out on free GPU time on Collab, so my resources are limited, but of course more detailed analysis is possible

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate
import torch
import numpy as np
import itertools
import pandas as pd


def run_single_experiment(model_name, batch_size, learning_rate, max_len, num_epochs=3, weight_decay=0.01):
    """Runs training for one set of hyperparameters and returns results dict."""

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # load dataset
    dataset = load_dataset("cardiffnlp/tweet_eval", "hate")

    # tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(example):
        return tokenizer(
            example["text"],
            padding="max_length",
            truncation=True,
            max_length=max_len
        )

    dataset = dataset.map(tokenize, batched=True)
    dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    # model
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2
    ).to(device)

    # metrics
    metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return metric.compute(predictions=preds, references=labels)

    # training arguments
    training_args = TrainingArguments(
        output_dir=f"./results-{model_name.replace('/', '-')}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        push_to_hub=False,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        compute_metrics=compute_metrics,
    )

    trainer.train()

    results_test = trainer.evaluate(dataset["test"])

    return {
        "model_name": model_name,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "max_len": max_len,
        "test_accuracy": results_test["eval_accuracy"],
        "test_loss": results_test["eval_loss"],
    }


def conduct_experiments_grid(model_name, param_grid):
    keys = list(param_grid.keys())
    values = list(param_grid.values())
    results = []

    print("Running experiments...")

    # iterate over all combinations
    for combo in itertools.product(*values):
        params = dict(zip(keys, combo))
        print(f"\n------ Running: {params} ------")

        res = run_single_experiment(
            model_name=model_name,
            batch_size=params.get("batch_size"),
            learning_rate=params.get("learning_rate"),
            max_len=params.get("max_len")
        )

        results.append(res)

    # convert to dataframe
    df = pd.DataFrame(results)
    print("\n------ Final Results ------")
    print(df)

    return df


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


**Model 1: FacebookAI/xlm-roberta-base with different parameters**

In [ ]:
param_grid = {
    "batch_size": [8],
    "learning_rate": [2e-5],
    "max_len": [128]
}

results_roberta_base_1 = conduct_experiments_grid("FacebookAI/xlm-roberta-base", param_grid)

results_roberta_base_1_df = pd.DataFrame(results_roberta_base_1)


Running experiments...

------ Running: {'batch_size': 8, 'learning_rate': 2e-05, 'max_len': 128} ------


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.487300,0.526832,0.740000
2,0.394900,0.572904,0.743000
3,0.332800,0.807057,0.762000



------ Final Results ------
                    model_name  batch_size  learning_rate  max_len  \
0  FacebookAI/xlm-roberta-base           8        0.00002      128   

   test_accuracy  test_loss  
0       0.511785   2.391567  


In [ ]:
param_grid = {
    "batch_size": [16],
    "learning_rate": [5e-5],
    "max_len": [256]
}

results_roberta_base_2 = conduct_experiments_grid("FacebookAI/xlm-roberta-base", param_grid)

results_roberta_base_2_df = pd.DataFrame(results_roberta_base_2)



Running experiments...

------ Running: {'batch_size': 16, 'learning_rate': 5e-05, 'max_len': 256} ------


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2970 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.681700,0.685069,0.573000
2,0.682600,0.625399,0.681000
3,0.618400,0.586505,0.700000



------ Final Results ------
                    model_name  batch_size  learning_rate  max_len  \
0  FacebookAI/xlm-roberta-base          16        0.00005      256   

   test_accuracy  test_loss  
0       0.474074   0.764259  


**Model 2: bert-base-uncased with different parameters**

In [ ]:
param_grid = {
    "batch_size": [8],
    "learning_rate": [2e-5],
    "max_len": [128]
}

results_bert_base_uncased_1 = conduct_experiments_grid("bert-base-uncased", param_grid)

results_bert_base_uncased_1_df = pd.DataFrame(results_bert_base_uncased_1)


Running experiments...

------ Running: {'batch_size': 8, 'learning_rate': 2e-05, 'max_len': 128} ------


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.438500,0.478074,0.758000
2,0.338500,0.587660,0.786000
3,0.214500,0.889065,0.793000



------ Final Results ------
          model_name  batch_size  learning_rate  max_len  test_accuracy  \
0  bert-base-uncased           8        0.00002      128       0.542088   

   test_loss  
0   2.317993  


In [ ]:
param_grid = {
    "batch_size": [16],
    "learning_rate": [5e-5],
    "max_len": [256]
}

results_bert_base_uncased_2 = conduct_experiments_grid("bert-base-uncased", param_grid)

results_bert_base_uncased_2_df = pd.DataFrame(results_bert_base_uncased_2)


Running experiments...

------ Running: {'batch_size': 16, 'learning_rate': 5e-05, 'max_len': 256} ------


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2970 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.475500,0.499067,0.744000
2,0.290600,0.497535,0.787000
3,0.168100,0.793591,0.789000



------ Final Results ------
          model_name  batch_size  learning_rate  max_len  test_accuracy  \
0  bert-base-uncased          16        0.00005      256       0.550168   

   test_loss  
0   2.454937  


**Model 3: microsoft/deberta-v3-small with different parameters**

In [ ]:
param_grid = {
    "batch_size": [8],
    "learning_rate": [2e-5],
    "max_len": [128]
}

results_deberta_v3_small_1 = conduct_experiments_grid("microsoft/deberta-v3-small", param_grid)

results_deberta_v3_small_1_df = pd.DataFrame(results_deberta_v3_small_1)


Running experiments...

------ Running: {'batch_size': 8, 'learning_rate': 2e-05, 'max_len': 128} ------


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.450300,0.499581,0.763000
2,0.362600,0.593264,0.761000
3,0.286700,0.745408,0.769000



------ Final Results ------
                   model_name  batch_size  learning_rate  max_len  \
0  microsoft/deberta-v3-small           8        0.00002      128   

   test_accuracy  test_loss  
0       0.504377   2.254863  


In [ ]:
param_grid = {
    "batch_size": [16],
    "learning_rate": [5e-5],
    "max_len": [256]
}

results_deberta_v3_small_2 = conduct_experiments_grid("microsoft/deberta-v3-small", param_grid)

results_deberta_v3_small_2_df = pd.DataFrame(results_deberta_v3_small_2)


Running experiments...

------ Running: {'batch_size': 16, 'learning_rate': 5e-05, 'max_len': 256} ------


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2970 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.490500,0.488108,0.775000
2,0.328100,0.546420,0.771000
3,0.227800,0.687324,0.786000



------ Final Results ------
                   model_name  batch_size  learning_rate  max_len  \
0  microsoft/deberta-v3-small          16        0.00005      256   

   test_accuracy  test_loss  
0       0.522222   2.335894  


**Final comparative table**

In [ ]:
#make a comparative table

import pandas as pd

all_results = pd.concat([
    results_deberta_v3_small_1_df.assign(experiment="deberta_v3_small_1"),
    results_deberta_v3_small_2_df.assign(experiment="deberra_v3_small_2"),

    results_bert_base_uncased_1_df.assign(experiment="bert_base_uncased_1"),
    results_bert_base_uncased_2_df.assign(experiment="bert_base_uncased_2"),

    results_roberta_base_1_df.assign(experiment="xlm_roberta_base_1"),
    results_roberta_base_2_df.assign(experiment="xlm_roberta_base_2")
], ignore_index=True)


all_results = all_results.rename(columns={
    "Value": "Test Accuracy",
    "test_loss": "Test Loss",
    "batch_size": "Batch Size",
    "learning_rate": "Learning Rate",
    "max_len": "Max Length",
})

print("\n------ Combined Table ------\n")
display(all_results)

print("\n------ Sorted by Test Accuracy ------\n")
sorted_results = all_results.sort_values(by="Test Accuracy", ascending=False)
display(sorted_results)



------ Combined Table ------



,Hyperparameter,Test Accuracy,Test Loss,experiment
0,Test Accuracy,0.504377,2.254863,deberta_v3_small_1
1,Test Accuracy,0.522222,2.335894,deberra_v3_small_2
2,Test Accuracy,0.542088,2.317993,bert_base_uncased_1
3,Test Accuracy,0.550168,2.454937,bert_base_uncased_2
4,Test Accuracy,0.511785,2.391567,xlm_roberta_base_1
5,Test Accuracy,0.474074,0.764259,xlm_roberta_base_2



------ Sorted by Test Accuracy ------



,Hyperparameter,Test Accuracy,Test Loss,experiment
3,Test Accuracy,0.550168,2.454937,bert_base_uncased_2
2,Test Accuracy,0.542088,2.317993,bert_base_uncased_1
1,Test Accuracy,0.522222,2.335894,deberra_v3_small_2
4,Test Accuracy,0.511785,2.391567,xlm_roberta_base_1
0,Test Accuracy,0.504377,2.254863,deberta_v3_small_1
5,Test Accuracy,0.474074,0.764259,xlm_roberta_base_2


# Conclusion:


In result of comparison of different pretrained models, the best results showed bert_base_uncased with params batch_size=16, learning_rate=5e-5,max_len=256, in which the main factor of umprovement i think is max_len. Overall, bert-sase_uncased showed better results then other models. In my particular case with this difficult "hate" dataset pretrained models showed worse result then MLP model with parameters which I varied. This is because dataset contains of  a lot of typos, irony, and other misleasing staff.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# load dataset
dataset = load_dataset("cardiffnlp/tweet_eval", "hate")

model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_hidden_states=True).to(device)
model.eval()

# embedding extraction part
def get_embedding(text, layer_idx):
    inputs = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        hidden_states = outputs.hidden_states

    layer = hidden_states[layer_idx]
    cls_embedding = layer[:, 0, :].squeeze().cpu().numpy()

    return cls_embedding

# layers
num_layers = len(model.encoder.layer) + 1
print("Total layers (including embedding layer):", num_layers)

layers_to_test = list(range(num_layers))

# building dataset
def build_dataset(split, layer_idx):
    texts = dataset[split]["text"]
    labels = dataset[split]["label"]

    embeddings = []
    for text in tqdm(texts, desc=f"{split} | layer {layer_idx}"):
        emb = get_embedding(text, layer_idx)
        embeddings.append(emb)

    X = np.vstack(embeddings)
    y = np.array(labels)
    return X, y

# train and evaluate
results = []

for layer_idx in layers_to_test:
    print(f"\n------ Extracting layer {layer_idx} ------")

    X_train, y_train = build_dataset("train", layer_idx)
    X_val, y_val = build_dataset("validation", layer_idx)
    X_test, y_test = build_dataset("test", layer_idx)

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, y_train)

    val_acc = accuracy_score(y_val, clf.predict(X_val))
    test_acc = accuracy_score(y_test, clf.predict(X_test))

    print(f"Layer {layer_idx}: val={val_acc:.4f}, test={test_acc:.4f}")

    results.append({
        "Layer": layer_idx,
        "Val Acc": val_acc,
        "Test Acc": test_acc
    })

df_results = pd.DataFrame(results).sort_values("Test Acc", ascending=False)
df_results


Device: cuda
Total layers (including embedding layer): 13

------ Extracting layer 0 ------


train | layer 0: 100%|██████████| 9000/9000 [01:33<00:00, 95.83it/s]
validation | layer 0: 100%|██████████| 1000/1000 [00:10<00:00, 95.55it/s]
test | layer 0: 100%|██████████| 2970/2970 [00:31<00:00, 94.89it/s]


Layer 0: val=0.5730, test=0.5785

------ Extracting layer 1 ------


train | layer 1: 100%|██████████| 9000/9000 [01:34<00:00, 95.64it/s]
validation | layer 1: 100%|██████████| 1000/1000 [00:10<00:00, 95.05it/s]
test | layer 1: 100%|██████████| 2970/2970 [00:30<00:00, 95.94it/s]


Layer 1: val=0.6510, test=0.5310

------ Extracting layer 2 ------


...

train | layer 11: 100%|██████████| 9000/9000 [01:33<00:00, 95.89it/s]
validation | layer 11: 100%|██████████| 1000/1000 [00:10<00:00, 96.66it/s]
test | layer 11: 100%|██████████| 2970/2970 [00:30<00:00, 96.03it/s]


Layer 11: val=0.6700, test=0.5182

------ Extracting layer 12 ------


train | layer 12: 100%|██████████| 9000/9000 [01:33<00:00, 95.77it/s]
validation | layer 12: 100%|██████████| 1000/1000 [00:10<00:00, 95.26it/s]
test | layer 12: 100%|██████████| 2970/2970 [00:31<00:00, 95.10it/s]


Layer 12: val=0.7270, test=0.5677


,Layer,Val Acc,Test Acc
0,0,0.573,0.578451
7,7,0.573,0.578451
12,12,0.727,0.567677
8,8,0.601,0.553199
6,6,0.600,0.546465
4,4,0.658,0.541414
9,9,0.619,0.535354
1,1,0.651,0.530976
2,2,0.650,0.528283
3,3,0.665,0.526599


In [ ]:
# Zero shot classifier models:

# 1) facebook/bart-large-mnli
# 2) MoritzLaurer/deberta-v3-base-zeroshot-v1
# 3) typeform/distilbert-base-uncased-mnli

from datasets import load_dataset
from transformers import pipeline
import evaluate
import numpy as np

# check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# labels: non-hate, hate
dataset = load_dataset("cardiffnlp/tweet_eval", "hate")

# getting label names as candidate labels for zero-shot
label_names = dataset["train"].features["label"].names
candidate_labels = list(label_names)

print("Candidate labels:", candidate_labels)

# zero shot classification pipeline
clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli",device=0)

# run zero-shot classification on the test set in batches
def predict_batch(examples):
    outputs = clf(examples["text"],candidate_labels=candidate_labels,multi_label=False)
    preds = []
    for out in outputs:
        top_label = out["labels"][0]
        preds.append(candidate_labels.index(top_label))
    return {"predictions": preds}

test_with_preds = dataset["test"].map(predict_batch,batched=True,batch_size=16)

# evaluating accuracy
metric = evaluate.load("accuracy")
accuracy = metric.compute(predictions=test_with_preds["predictions"],references=test_with_preds["label"])
print("Zero-shot accuracy on hate:", accuracy)


Device: cuda
Candidate labels: ['non-hate', 'hate']


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Map:   0%|          | 0/2970 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Zero-shot accuracy on hate: {'accuracy': 0.5740740740740741}


In [ ]:
# Model: MoritzLaurer/deberta-v3-base-zeroshot-v1

import torch
from datasets import load_dataset
from transformers import pipeline
import evaluate
import numpy as np

device = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if device == 0 else "CPU")

dataset = load_dataset("cardiffnlp/tweet_eval", "hate")

# labels: non-hate, hate
candidate_labels = dataset["train"].features["label"].names
print("Candidate labels:", candidate_labels)

model_name = "MoritzLaurer/deberta-v3-base-zeroshot-v1"

clf = pipeline(task="zero-shot-classification",model=model_name,device=device)

# batch prediction
def predict_batch(examples):
    outputs = clf(examples["text"],candidate_labels=candidate_labels,multi_label=False)

    preds = [
        candidate_labels.index(out["labels"][0])
        for out in outputs
    ]
    return {"predictions": preds}

test_with_preds = dataset["test"].map(predict_batch,batched=True,batch_size=8)

# evaluation
metric = evaluate.load("accuracy")
accuracy = metric.compute(predictions=test_with_preds["predictions"],references=test_with_preds["label"])

print(f"Zero-shot Accuracy ({model_name}): {accuracy['accuracy']:.4f}")


Device: GPU
Candidate labels: ['non-hate', 'hate']


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Device set to use cuda:0


Map:   0%|          | 0/2970 [00:00<?, ? examples/s]

Zero-shot Accuracy (MoritzLaurer/deberta-v3-base-zeroshot-v1): 0.6051


In [ ]:
# Model: "typeform/distilbert-base-uncased-mnli"
import torch
from datasets import load_dataset
from transformers import pipeline
import evaluate
import numpy as np

device = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if device == 0 else "CPU")

dataset = load_dataset("cardiffnlp/tweet_eval", "hate")

# labels: non-hate, hate
candidate_labels = dataset["train"].features["label"].names
print("Candidate labels:", candidate_labels)

model_name = "typeform/distilbert-base-uncased-mnli"

clf = pipeline(
    task="zero-shot-classification",model=model_name, tokenizer=model_name, device=device)

def predict_batch(examples):
    outputs = clf(examples["text"],candidate_labels=candidate_labels,multi_label=False)
    preds = [candidate_labels.index(out["labels"][0]) for out in outputs]
    return {"predictions": preds}

test_with_preds = dataset["test"].map(predict_batch,batched=True,batch_size=8)

metric = evaluate.load("accuracy")
accuracy = metric.compute(predictions=test_with_preds["predictions"],references=test_with_preds["label"])

print(f"Zero-shot Accuracy ({model_name}): {accuracy['accuracy']:.4f}")


Device: GPU
Candidate labels: ['non-hate', 'hate']


config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


Map:   0%|          | 0/2970 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Zero-shot Accuracy (typeform/distilbert-base-uncased-mnli): 0.5879
